# Training monitor
Follow the overnight training while `train_road_obstacles.ipynb` runs — or after VS Code was closed (training keeps running in the Docker container). It reads what Ultralytics writes after **every epoch**.

Same kernel as the training notebook: **Existing Jupyter Server** `http://localhost:8888/?token=smartcar` → Python 3. Run the first cell for a snapshot, the second one for an auto-refresh every minute (■ to stop).

In [ ]:
import csv, subprocess, time
from pathlib import Path

import matplotlib.pyplot as plt
import yaml
from IPython.display import clear_output

RUN = Path("/workspace/training/runs/road_hazards")
P, R, M50, M = "metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"


def snapshot():
    results, args = RUN / "results.csv", RUN / "args.yaml"
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
    print(f"{time.strftime('%H:%M:%S')} · GPU (use, memory, total, °C): {gpu or 'n/a'}")
    if not results.exists():
        print("No epoch finished yet — downloads take ~20 min the first time, then one epoch ≈ 11 min.")
        return
    with open(results) as f:
        rows = [{k.strip(): float(v) for k, v in r.items()} for r in csv.DictReader(f)]
    if not rows:
        print("No epoch finished yet.")
        return
    budget_h = (yaml.safe_load(args.read_text()) or {}).get("time") if args.exists() else None
    last, best = rows[-1], max(rows, key=lambda r: r[M])
    elapsed_h = last.get("time", 0) / 3600
    print(f"epochs done: {int(last['epoch'])} · elapsed {elapsed_h:.1f} h"
          + (f" of {budget_h} h · ≈ {max(0, budget_h - elapsed_h):.1f} h left" if budget_h else ""))
    print(f"last epoch : precision {last[P]:.3f} · recall {last[R]:.3f} · mAP50 {last[M50]:.3f} · mAP50-95 {last[M]:.3f}")
    print(f"best so far: mAP50-95 {best[M]:.3f} at epoch {int(best['epoch'])}  (saved as best.pt)")
    epochs = [r["epoch"] for r in rows]
    fig, (a, b) = plt.subplots(1, 2, figsize=(14, 4))
    for col in [c for c in rows[0] if c.startswith("train/") and c.endswith("loss")]:
        a.plot(epochs, [r[col] for r in rows], label=col.split("/")[1])
    a.set_title("training losses"); a.set_xlabel("epoch"); a.legend()
    for col, label in ((M50, "mAP50"), (M, "mAP50-95"), (P, "precision"), (R, "recall")):
        b.plot(epochs, [r[col] for r in rows], label=label)
    b.set_title("validation"); b.set_xlabel("epoch"); b.set_ylim(0, 1); b.legend()
    plt.show()
    print(f"{'epoch':>5} {'precision':>9} {'recall':>7} {'mAP50':>6} {'mAP50-95':>8}")
    for r in rows[-8:]:
        print(f"{int(r['epoch']):>5} {r[P]:>9.3f} {r[R]:>7.3f} {r[M50]:>6.3f} {r[M]:>8.3f}")


snapshot()

In [ ]:
# Auto-refresh every minute (press ■ to stop)
while True:
    clear_output(wait=True)
    snapshot()
    time.sleep(60)